# Tutorial dbt: Arquitectura Medallón con DuckDB

En este tutorial construiremos un pipeline de datos completo usando **dbt** (data build tool) con **DuckDB** como motor analítico.

## ¿Qué vamos a construir?

Un pipeline de 3 capas (arquitectura medallón) con datos sintéticos de estadísticas gubernamentales mexicanas:

| Capa | Esquema | Propósito |
|------|---------|----------|
| **Staging (Bronze)** | `staging` | Limpieza y normalización de datos crudos |
| **Intermediate (Silver)** | `intermediate` | Transformaciones de negocio, joins, claves sustitutas |
| **Mart (Gold)** | `mart` | Tablas dimensionales y de hechos listas para análisis |

## Fuentes de datos

- **Censo Económico (CE)**: 630 filas con 10 indicadores económicos por entidad/municipio/actividad SCIAN
- **Presupuesto DOF**: 76 filas de asignaciones presupuestales federales con datos sucios

## Requisitos previos

- Python 3.10+
- `uv` como gestor de paquetes (o pip)
- Familiaridad básica con SQL

## Estructura de archivos

```
notebook/
├── tutorial_dbt.ipynb        ← Este notebook
├── datos_tutorial.duckdb     ← Base de datos pre-generada
├── mi_proyecto_dbt/          ← Tu proyecto (lo crearás aquí)
└── soluciones/               ← Proyecto resuelto de referencia
```

> **Documentación**: [¿Qué es dbt?](https://docs.getdbt.com/docs/introduction) · [Arquitectura de proyectos dbt](https://docs.getdbt.com/best-practices/how-we-structure/1-guide-overview)


---
## Sección 0: Preparación del entorno

### Instalación de dependencias

Necesitamos `dbt-duckdb` (que incluye dbt-core) y `dbt-utils` como paquete de dbt.

In [ ]:
# Si usas uv (recomendado):
# !uv pip install dbt-duckdb duckdb

### Exploración de los datos crudos

Antes de transformar, hay que entender qué tenemos. Conectémonos a la base de datos y veamos las tablas disponibles.

In [ ]:
import duckdb

con = duckdb.connect("datos_tutorial.duckdb", read_only=True)

# Tablas disponibles
con.sql("SHOW TABLES").show()

In [ ]:
# Conteo de filas por tabla
for tabla in [
    "ce_datos",
    "ce_catalogos_actividades",
    "ce_catalogos_entidades_municipios",
    "ce_diccionarios_datos",
    "dof_presupuesto",
]:
    conteo = con.sql(f"SELECT COUNT(*) AS filas FROM {tabla}").fetchone()[0]  # type: ignore[index]
    print(f"{tabla}: {conteo} filas")

In [ ]:
# Censo Económico: estructura y primeras filas
con.sql("SELECT * FROM ce_datos LIMIT 5").show()

In [ ]:
# Catálogo de actividades SCIAN
con.sql("SELECT * FROM ce_catalogos_actividades").show()

In [ ]:
# Catálogo geográfico
con.sql("SELECT * FROM ce_catalogos_entidades_municipios").show()

In [ ]:
# Presupuesto DOF: observa los nombres sucios de entidad
con.sql("SELECT DISTINCT entidad FROM dof_presupuesto ORDER BY entidad").show()

### Datos sucios identificados

Observa los problemas que nuestro pipeline debe resolver:

In [ ]:
# Problema 1: Espacios extra en strings
con.sql("""
    SELECT entidad, LENGTH(entidad) AS longitud
    FROM ce_datos
    WHERE entidad <> TRIM(entidad)
    LIMIT 5
""").show()

# Problema 2: Strings vacíos vs NULL para nivel nacional
con.sql("""
    SELECT entidad, municipio, COUNT(*) AS filas
    FROM ce_datos
    WHERE entidad IN ('', '  ', ' ')
    GROUP BY entidad, municipio
""").show()

# Problema 3: Nombres inconsistentes en DOF (acentos, mayúsculas, asteriscos)
con.sql("""
    SELECT entidad, es_total, es_consolidado
    FROM dof_presupuesto
    WHERE entidad LIKE '%*%' OR entidad = 'TOTAL' OR entidad LIKE '%Consolidado%'
    LIMIT 10
""").show()

In [ ]:
con.close()

### Crear la estructura de directorios del proyecto

In [ ]:
!mkdir -p mi_proyecto_dbt/models/staging
!mkdir -p mi_proyecto_dbt/models/intermediate
!mkdir -p mi_proyecto_dbt/models/mart
!mkdir -p mi_proyecto_dbt/macros
!mkdir -p mi_proyecto_dbt/seeds
print("Estructura de directorios creada.")

---
## Sección 1: Inicialización del proyecto dbt

Un proyecto dbt necesita al menos dos archivos de configuración:
- `dbt_project.yml`: define el nombre, rutas y configuraciones del proyecto
- `profiles.yml`: define la conexión a la base de datos

> **Nota**: En la práctica real, estos archivos se crean con un editor de texto o con `dbt init`. Aquí usamos `%%writefile` para mantener todo en el notebook.

> **Documentación**: [Estructura de un proyecto dbt](https://docs.getdbt.com/docs/build/projects) · [Comando `dbt init`](https://docs.getdbt.com/reference/commands/init)


### Ejercicio 1.1: Archivo de proyecto (`dbt_project.yml`)

Este es el archivo principal de configuración. Define el nombre del proyecto, la versión y las rutas donde dbt buscará modelos, seeds y macros.

> **Documentación**: [Referencia de `dbt_project.yml`](https://docs.getdbt.com/reference/dbt_project.yml)


In [ ]:
%%writefile mi_proyecto_dbt/dbt_project.yml
name: tutorial_dbt
version: "0.1.0"
config-version: 2
profile: tutorial_dbt

model-paths: ["models"]
seed-paths: ["seeds"]
macro-paths: ["macros"]

### Ejercicio 1.2: Perfil de conexión (`profiles.yml`)

El perfil define cómo conectarse a la base de datos. Usamos DuckDB con la ruta relativa a `datos_tutorial.duckdb`.

- `path`: ruta al archivo DuckDB (relativa al directorio del proyecto)
- `schema`: esquema por defecto donde se crearán los modelos
- `threads`: número de hilos de ejecución paralela

> En producción, `profiles.yml` suele vivir en `~/.dbt/` y NO se versiona en git. Aquí lo ponemos en el proyecto por simplicidad.

> **Documentación**: [Referencia de `profiles.yml`](https://docs.getdbt.com/docs/core/connect-data-platform/profiles.yml) · [Configuración de DuckDB](https://docs.getdbt.com/docs/core/connect-data-platform/duckdb-setup)


In [ ]:
%%writefile mi_proyecto_dbt/profiles.yml
tutorial_dbt:
  target: dev
  outputs:
    dev:
      type: duckdb
      path: "../datos_tutorial.duckdb"
      schema: mart
      threads: 1

### Ejercicio 1.3: Paquetes (`packages.yml`)

dbt tiene un ecosistema de paquetes reutilizables. Usaremos `dbt_utils` que provee macros útiles como `generate_surrogate_key` para crear claves sustitutas.

> **Documentación**: [Paquetes dbt](https://docs.getdbt.com/docs/build/packages) · [dbt-utils en dbt Hub](https://hub.getdbt.com/dbt-labs/dbt_utils/latest/)


In [ ]:
%%writefile mi_proyecto_dbt/packages.yml
packages:
  - package: dbt-labs/dbt_utils
    version: ">=1.1.0"

El command `dbt deps` instala a todas las dependencias que definimos (el `dbt_utils` en este caso).

> **Documentación**: [Comando `dbt deps`](https://docs.getdbt.com/reference/commands/deps)


In [ ]:
# Instalar dependencias de dbt
!cd mi_proyecto_dbt && uv run dbt deps

### Validar la conexión

`dbt debug` verifica que la configuración y la conexión a la base de datos sean correctas.

> **Documentación**: [Comando `dbt debug`](https://docs.getdbt.com/reference/commands/debug)


In [ ]:
!cd mi_proyecto_dbt && uv run dbt debug

---
## Sección 2: Sources (fuentes de datos)

En dbt, las **sources** declaran las tablas crudas que ya existen en la base de datos. Usar `{{ source("nombre", "tabla") }}` en vez de nombres directos de tabla nos da:

1. **Documentación**: las fuentes aparecen en el catálogo de dbt
2. **Linaje**: dbt sabe de dónde vienen los datos
3. **Freshness**: se puede verificar qué tan recientes son los datos
4. **Refactorización fácil**: si la tabla cambia de nombre, se actualiza en un solo lugar

> **Documentación**: [Sources (fuentes)](https://docs.getdbt.com/docs/build/sources) · [Función `source()`](https://docs.getdbt.com/reference/dbt-jinja-functions/source) · [Source freshness](https://docs.getdbt.com/docs/build/sources#source-data-freshness)


### Ejercicio 2.1: Declarar las fuentes (`sources.yml`)

Declaramos las 5 tablas crudas del esquema `main` como fuentes del proyecto.

In [ ]:
%%writefile mi_proyecto_dbt/models/sources.yml
version: 2

sources:
  - name: tutorial
    description: "Tablas crudas sintéticas para el tutorial de dbt"
    schema: main
    tables:
      - name: ce_datos
        description: "Microdatos del Censo Económico — tabla ancha con 10 indicadores por año × entidad × municipio × actividad"
      - name: ce_catalogos_actividades
        description: "Catálogo de actividades económicas SCIAN con jerarquía sector/subsector/rama"
      - name: ce_catalogos_entidades_municipios
        description: "Catálogo geográfico de 5 entidades y 15 municipios"
      - name: ce_diccionarios_datos
        description: "Diccionario de datos CE — descripciones de los 10 indicadores + filas de metadatos"
      - name: dof_presupuesto
        description: "Presupuesto federal DOF — asignaciones por entidad con desgloses mensuales"

---
## Sección 3: Staging (limpieza y normalización)

La capa de staging es la primera transformación. Sus responsabilidades:

- **Limpiar strings**: `TRIM()`, `NULLIF()` para convertir vacíos en NULL
- **Estandarizar tipos**: casteos, formateos
- **Renombrar columnas**: nombres consistentes
- **NO agregar lógica de negocio**: eso va en capas posteriores

### Convención de nombres

Los modelos de staging siguen el patrón: `stg_<fuente>__<entidad>.sql` (doble guión bajo entre fuente y entidad).

> **Documentación**: [Modelos dbt](https://docs.getdbt.com/docs/build/models) · [Guía de staging](https://docs.getdbt.com/best-practices/how-we-structure/2-staging)


### Ejercicio 3.1: Staging del Censo Económico (`stg_ce__datos.sql`)

Este es el modelo más importante de staging. Limpia los datos crudos del CE:

- `TRIM()` elimina espacios extra de las claves geográficas
- `NULLIF(valor, '')` convierte strings vacíos en NULL
- El `CASE` clasifica cada fila en su nivel geográfico (nacional/entidad/municipio)
- Los 10 indicadores pasan sin transformación

In [ ]:
%%writefile mi_proyecto_dbt/models/staging/stg_ce__datos.sql
-- Staging del Censo Económico: limpieza de strings, clasificación de nivel geográfico,
-- y paso de los 10 indicadores sin transformación.
with
    fuente as (select * from {{ source("tutorial", "ce_datos") }}),

    limpio as (
        select
            id as id_fuente,
            anio,
            nullif(trim(entidad), '') as cve_ent,
            nullif(trim(municipio), '') as cve_mun,
            nullif(trim(codigo), '') as codigo_actividad,
            case
                when trim(entidad) = '' and trim(municipio) = ''
                then 'nacional'
                when trim(entidad) <> '' and trim(municipio) = ''
                then 'entidad'
                when trim(entidad) <> '' and trim(municipio) <> ''
                then 'municipio'
            end as nivel_geografico,

            -- 10 indicadores: se pasan tal cual
            ue,
            h001a,
            h000a,
            i000a,
            j000a,
            k000a,
            a111a,
            m000a,
            p000a,
            q000a
        from fuente
    )

select *
from limpio

### Ejercicio 3.2: Staging de actividades (`stg_ce__actividades.sql`)

Limpieza simple del catálogo SCIAN: solo `TRIM()` en los campos de texto.

In [ ]:
%%writefile mi_proyecto_dbt/models/staging/stg_ce__actividades.sql
-- Staging del catálogo de actividades económicas SCIAN.
-- Limpieza de strings y paso de campos.
with fuente as (select * from {{ source("tutorial", "ce_catalogos_actividades") }})

select
    trim(codigo) as codigo,
    trim(descripcion) as descripcion,
    trim(clasificador) as clasificador
from fuente

### Ejercicio 3.3: Staging de entidades y municipios (`stg_ce__entidades_municipios.sql`)

Limpieza del catálogo geográfico.

In [ ]:
%%writefile mi_proyecto_dbt/models/staging/stg_ce__entidades_municipios.sql
-- Staging del catálogo geográfico de entidades y municipios.
-- Limpieza de strings y selección de columnas relevantes.
with
    fuente as (select * from {{ source("tutorial", "ce_catalogos_entidades_municipios") }})

select
    trim(cvegeo) as cvegeo,
    trim(cve_ent) as cve_ent,
    trim(nombre_entidad) as nombre_entidad,
    trim(cve_mun) as cve_mun,
    trim(nombre_municipio) as nombre_municipio
from fuente

### Ejercicio 3.4: Staging del presupuesto DOF (`stg_dof__presupuesto.sql`)

Este es el modelo de staging más complejo. Los datos del DOF tienen:
- Nombres de entidad con espacios extra, acentos, asteriscos y capitalización inconsistente
- Filas de "TOTAL" y "Consolidado Nacional" que son basura para nuestros fines

Usamos un bloque Jinja `{% set %}` para definir una cadena de `regexp_replace` que normaliza el texto:
1. Normaliza espacios múltiples
2. Quita asteriscos iniciales
3. Elimina acentos y ñ
4. Convierte a minúsculas

Luego un `VALUES` CTE mapea los nombres normalizados a claves de entidad (`cve_ent`).

> En la sección 7 refactorizaremos esta lógica a una macro reutilizable.

> **Documentación**: [Jinja y macros en dbt](https://docs.getdbt.com/docs/build/jinja-macros) · [Referencia de Jinja](https://docs.getdbt.com/reference/dbt-jinja-functions) · [`{% set %}` en Jinja](https://jinja.palletsprojects.com/en/3.1.x/templates/#assignments)


In [ ]:
%%writefile mi_proyecto_dbt/models/staging/stg_dof__presupuesto.sql
-- Staging del presupuesto DOF: normaliza nombres de entidad a cve_ent
-- mediante cadena de regexp_replace y mapeo inline de 5 estados.
-- Las filas que no mapean (junk: totales, consolidados) quedan con cve_ent NULL.

{%- set normalizar_entidad -%}
regexp_replace(
    regexp_replace(
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    regexp_replace(
                        regexp_replace(
                            regexp_replace(
                                lower(regexp_replace(trim(entidad), '\s+', ' ', 'g')),
                                '^\*+\s*', ''
                            ),
                            'á', 'a'
                        ),
                        'é', 'e'
                    ),
                    'í', 'i'
                ),
                'ó', 'o'
            ),
            'ú', 'u'
        ),
        'ü', 'u'
    ),
    'ñ', 'ni'
)
{%- endset %}

with
    fuente as (select * from {{ source("tutorial", "dof_presupuesto") }}),

    mapeo_entidades(nombre_normalizado, cve_ent) as (
        values
            ('ciudad de mexico', '09'),
            ('guanajuato', '11'),
            ('jalisco', '14'),
            ('nuevo leon', '19'),
            ('puebla', '21')
    ),

    con_normalizacion as (
        select *, {{ normalizar_entidad }} as entidad_normalizada from fuente
    ),

    mapeado as (
        select
            f.id as id_fuente,
            f.anio,
            f.ramo,
            f.anexo,
            trim(f.fondo) as fondo,
            trim(f.entidad) as entidad_original,
            f.entidad_normalizada,
            m.cve_ent,
            f.es_total,
            f.es_consolidado,
            f.anual,
            f.enero,
            f.febrero,
            f.marzo,
            f.abril,
            f.mayo,
            f.junio,
            f.julio,
            f.agosto,
            f.septiembre,
            f.octubre,
            f.noviembre,
            f.diciembre
        from con_normalizacion as f
        left join mapeo_entidades as m on f.entidad_normalizada = m.nombre_normalizado
    )

select *
from mapeado

### Ejercicio 3.5: Documentación y tests de staging (`_stg__models.yml`)

Los archivos YAML de documentación definen:
- **Descripciones** de cada modelo y columna
- **Tests** que dbt ejecuta para validar la calidad de los datos

Tests comunes:
- `unique`: sin duplicados
- `not_null`: sin valores nulos
- `accepted_values`: solo valores permitidos (nota: usa `arguments:` como wrapper)

> **Documentación**: [Tests de datos](https://docs.getdbt.com/docs/build/data-tests) · [Tests genéricos (`unique`, `not_null`, etc.)](https://docs.getdbt.com/reference/resource-properties/data-tests#generic-data-tests) · [Propiedades de modelos (YAML)](https://docs.getdbt.com/reference/model-properties)


In [ ]:
%%writefile mi_proyecto_dbt/models/staging/_stg__models.yml
version: 2

models:
  - name: stg_ce__datos
    description: "Datos del Censo Económico con nivel geográfico clasificado y 10 indicadores"
    columns:
      - name: id_fuente
        description: "ID original de la tabla fuente"
      - name: anio
        description: "Año censal"
        tests:
          - not_null
      - name: nivel_geografico
        description: "Clasificación: nacional, entidad o municipio"
        tests:
          - accepted_values:
              arguments:
                values: ["nacional", "entidad", "municipio"]

  - name: stg_ce__actividades
    description: "Catálogo SCIAN de actividades económicas"
    columns:
      - name: codigo
        description: "Código SCIAN de la actividad"
        tests:
          - unique
          - not_null

  - name: stg_ce__entidades_municipios
    description: "Catálogo geográfico de entidades y municipios"
    columns:
      - name: cvegeo
        description: "Clave geoestadística de 5 caracteres (cve_ent + cve_mun)"
        tests:
          - unique
          - not_null

  - name: stg_dof__presupuesto
    description: "Presupuesto DOF con entidad normalizada a cve_ent"
    columns:
      - name: id_fuente
        description: "ID original de la tabla fuente"
      - name: anio
        description: "Año fiscal"
        tests:
          - not_null
      - name: entidad_original
        description: "Nombre original de la entidad (para auditoría)"
      - name: cve_ent
        description: "Clave de entidad normalizada (NULL para filas junk)"

### Ejecutar y validar staging

In [ ]:
# Ejecutar los modelos de staging
!cd mi_proyecto_dbt && uv run dbt run --select staging

In [ ]:
# Ejecutar los tests de staging
!cd mi_proyecto_dbt && uv run dbt test --select staging

---
## Sección 4: Materialización y esquemas

dbt puede materializar modelos de diferentes formas:

| Tipo | Cuándo usar | Ventaja |
|------|-------------|--------|
| **view** | Staging, datos pequeños | Sin costo de almacenamiento, siempre frescos |
| **table** | Marts, datos consultados frecuentemente | Lectura rápida, datos precalculados |
| **incremental** | Tablas grandes que crecen | Solo procesa datos nuevos |

### Esquemas personalizados

Por defecto, dbt concatena el esquema del perfil con el `+schema` del modelo (ej: `mart_staging`). Nosotros queremos esquemas limpios (`staging`, `intermediate`, `mart`), así que necesitamos una macro personalizada.

> **Documentación**: [Materializaciones](https://docs.getdbt.com/docs/build/materializations) · [Modelos incrementales](https://docs.getdbt.com/docs/build/incremental-models) · [Esquemas personalizados](https://docs.getdbt.com/docs/build/custom-schemas)


### Ejercicio 4.1: Actualizar `dbt_project.yml` con materialización por capa

Configuramos cada capa con su esquema y tipo de materialización.

In [ ]:
%%writefile mi_proyecto_dbt/dbt_project.yml
name: tutorial_dbt
version: "0.1.0"
config-version: 2
profile: tutorial_dbt

model-paths: ["models"]
seed-paths: ["seeds"]
macro-paths: ["macros"]

models:
  tutorial_dbt:
    staging:
      +schema: staging
      +materialized: view
    intermediate:
      +schema: intermediate
      +materialized: view
    mart:
      +schema: mart
      +materialized: table

### Ejercicio 4.2: Macro `generate_schema_name`

Esta macro sobreescribe el comportamiento por defecto de dbt. Sin ella, el esquema sería `mart_staging` en vez de solo `staging`.

La lógica: si hay un `custom_schema_name` configurado, usarlo tal cual; si no, usar el esquema del target.

> **Documentación**: [Macro `generate_schema_name`](https://docs.getdbt.com/docs/build/custom-schemas#how-does-dbt-generate-a-models-schema-name)


In [ ]:
%%writefile mi_proyecto_dbt/macros/generate_schema_name.sql
{% macro generate_schema_name(custom_schema_name, node) -%}
    {%- if custom_schema_name is none -%} {{ target.schema }}
    {%- else -%} {{ custom_schema_name | trim }}
    {%- endif -%}
{%- endmacro %}

---
## Sección 5: Intermediate (transformaciones de negocio)

La capa intermediate agrega lógica de negocio:

- **Joins** entre tablas de staging
- **Claves sustitutas** (surrogate keys) para identificar registros únicamente
- **Denormalización** de jerarquías
- **Filtrado** de datos basura

### `generate_surrogate_key`

Esta macro de `dbt_utils` genera un hash MD5 a partir de una lista de columnas. Es útil para crear claves primarias cuando no existe una natural.

> **Documentación**: [Función `ref()`](https://docs.getdbt.com/reference/dbt-jinja-functions/ref) · [Guía de intermediate](https://docs.getdbt.com/best-practices/how-we-structure/3-intermediate) · [`generate_surrogate_key`](https://github.com/dbt-labs/dbt-utils#generate_surrogate_key-source)


### Ejercicio 5.1: Intermediate CE con jerarquía SCIAN (`int_ce__datos.sql`)

Este modelo:
1. Hace JOIN con el catálogo de actividades para obtener el `nivel_scian`
2. Genera una clave sustituta sobre el grano completo
3. Denormaliza la jerarquía SCIAN (sector, subsector, rama)
4. Maneja los sectores compuestos del SCIAN (31-33, 48-49)

In [ ]:
%%writefile mi_proyecto_dbt/models/intermediate/int_ce__datos.sql
-- Datos del Censo Económico con jerarquía SCIAN denormalizada.
-- Incluye nivel_scian derivado del catálogo de actividades.
-- Grano: anio × nivel_geografico × cve_ent × cve_mun × codigo_actividad.
with
    fuente as (select * from {{ ref("stg_ce__datos") }}),

    actividades as (
        select codigo, lower(clasificador) as nivel_scian
        from {{ ref("stg_ce__actividades") }}
    )

select
    {{
        dbt_utils.generate_surrogate_key(
            [
                "anio",
                "nivel_geografico",
                "cve_ent",
                "cve_mun",
                "codigo_actividad",
            ]
        )
    }} as censo_economico_sk,

    f.anio,
    f.nivel_geografico,
    f.cve_ent,
    f.cve_mun,
    f.codigo_actividad,

    -- Nivel SCIAN del catálogo de actividades
    a.nivel_scian,

    -- Jerarquía SCIAN denormalizada con sectores compuestos
    case
        when left(f.codigo_actividad, 2) in ('31', '32', '33')
        then '31-33'
        when left(f.codigo_actividad, 2) in ('48', '49')
        then '48-49'
        else left(f.codigo_actividad, 2)
    end as codigo_sector,

    case
        when a.nivel_scian in ('subsector', 'rama', 'subrama', 'clase')
        then left(f.codigo_actividad, 3)
    end as codigo_subsector,

    case
        when a.nivel_scian in ('rama', 'subrama', 'clase')
        then left(f.codigo_actividad, 4)
    end as codigo_rama,

    -- 10 indicadores pivotados
    f.ue,
    f.h001a,
    f.h000a,
    f.i000a,
    f.j000a,
    f.k000a,
    f.a111a,
    f.m000a,
    f.p000a,
    f.q000a

from fuente as f
inner join actividades as a on f.codigo_actividad = a.codigo

### Ejercicio 5.2: Intermediate DOF filtrado (`int_dof__presupuesto.sql`)

Filtra las filas basura (totales, consolidados, entidades no mapeadas) y agrega una clave sustituta.

In [ ]:
%%writefile mi_proyecto_dbt/models/intermediate/int_dof__presupuesto.sql
-- Filtra presupuesto DOF válido: solo filas con entidad mapeada,
-- excluyendo totales y consolidados.
-- Agrega clave sustituta sobre año × ramo × anexo × fondo × entidad.
with
    fuente as (
        select *
        from {{ ref("stg_dof__presupuesto") }}
        where cve_ent is not null and es_total = false and es_consolidado = false
    )

select
    {{
        dbt_utils.generate_surrogate_key(
            ["anio", "ramo", "anexo", "fondo", "cve_ent"]
        )
    }} as presupuesto_sk, *
from fuente

### Ejercicio 5.3: Tests de intermediate (`_int__models.yml`)

In [ ]:
%%writefile mi_proyecto_dbt/models/intermediate/_int__models.yml
version: 2

models:
  - name: int_ce__datos
    description: >
      Datos del Censo Económico con jerarquía SCIAN denormalizada.
      Grano: anio × nivel_geografico × cve_ent × cve_mun × codigo_actividad.
    columns:
      - name: censo_economico_sk
        description: "Clave sustituta sobre el grano completo"
        tests:
          - unique
          - not_null
      - name: nivel_geografico
        description: "Nivel geográfico: nacional, entidad o municipio"
        tests:
          - not_null
          - accepted_values:
              arguments:
                values: ["nacional", "entidad", "municipio"]
      - name: nivel_scian
        description: "Nivel SCIAN: sector, subsector o rama"
        tests:
          - not_null
          - accepted_values:
              arguments:
                values: ["sector", "subsector", "rama"]
      - name: codigo_sector
        description: "Código del sector SCIAN (maneja compuestos: 31-33)"
        tests:
          - not_null

  - name: int_dof__presupuesto
    description: "Presupuesto DOF filtrado: solo entidades válidas, sin totales ni consolidados"
    columns:
      - name: presupuesto_sk
        description: "Clave sustituta: anio × ramo × anexo × fondo × cve_ent"
        tests:
          - not_null

### Ejecutar y validar intermediate

In [ ]:
!cd mi_proyecto_dbt && uv run dbt run --select intermediate

In [ ]:
!cd mi_proyecto_dbt && uv run dbt test --select intermediate

---
## Sección 6: Marts (tablas dimensionales y de hechos)

La capa de marts produce las tablas finales listas para consumo en herramientas de BI o análisis directo. Seguimos el patrón dimensional:

- **Dimensiones** (`dim_`): tablas descriptivas (entidades, actividades, indicadores)
- **Hechos** (`fct_`): tablas de métricas/medidas (censo económico, presupuesto mensual)

Estas tablas se materializan como `table` para lecturas rápidas.

> **Documentación**: [Guía de marts](https://docs.getdbt.com/best-practices/how-we-structure/4-marts) · [Modelado dimensional](https://docs.getdbt.com/terms/dimensional-modeling)


### Ejercicio 6.1: Dimensión de entidades (`dim_entidades.sql`)

Una tabla simple con las 5 entidades federativas. `SELECT DISTINCT` asegura una fila por entidad.

In [ ]:
%%writefile mi_proyecto_dbt/models/mart/dim_entidades.sql
-- Dimensión de 5 entidades federativas del tutorial.
select distinct cve_ent, nombre_entidad
from {{ ref("stg_ce__entidades_municipios") }}
order by cve_ent

### Ejercicio 6.2: Dimensión de actividades económicas (`dim_actividades_economicas.sql`)

Jerarquía SCIAN con navegación padre-hijo para drill-down en BI.

Puntos clave:
- `codigo_padre`: apunta al nivel inmediatamente superior (NULL para sectores)
- `codigo_sector`: siempre presente, maneja el compuesto 31-33
- `descripcion_sector`: label del sector para dashboards

In [ ]:
%%writefile mi_proyecto_dbt/models/mart/dim_actividades_economicas.sql
-- Dimensión SCIAN con jerarquía para drill-down en BI.
-- Incluye codigo_padre para navegación jerárquica y descripcion_sector para labels.
-- Maneja el sector compuesto 31-33 del SCIAN.
with
    actividades as (
        select codigo, descripcion, lower(clasificador) as nivel
        from {{ ref("stg_ce__actividades") }}
    ),

    con_sector as (
        select
            a.codigo,
            a.descripcion,
            a.nivel,
            case
                when left(a.codigo, 2) in ('31', '32', '33')
                then '31-33'
                else left(a.codigo, 2)
            end as codigo_sector,
            case
                when a.nivel = 'sector'
                then null
                when a.nivel = 'subsector'
                then
                    case
                        when left(a.codigo, 2) in ('31', '32', '33')
                        then '31-33'
                        else left(a.codigo, 2)
                    end
                when a.nivel = 'rama'
                then left(a.codigo, 3)
            end as codigo_padre
        from actividades as a
    )

select
    cs.codigo,
    cs.descripcion,
    cs.nivel,
    cs.codigo_padre,
    cs.codigo_sector,
    sec.descripcion as descripcion_sector
from con_sector as cs
left join actividades as sec on cs.codigo_sector = sec.codigo and sec.nivel = 'sector'
order by cs.codigo

### Ejercicio 6.3: Tabla de hechos del Censo Económico (`fct_censo_economico.sql`)

Tabla principal de hechos con los 10 indicadores como columnas (formato ancho, eficiente para DuckDB columnar).

Usa `post_hook` para crear índices después de materializar la tabla, mejorando el rendimiento de consultas filtradas.

> **Documentación**: [Pre-hook y post-hook](https://docs.getdbt.com/reference/resource-configs/pre-hook-post-hook) · [Bloque `config()`](https://docs.getdbt.com/reference/dbt-jinja-functions/config)


In [ ]:
%%writefile mi_proyecto_dbt/models/mart/fct_censo_economico.sql
-- Censo Económico pivotado con todos los niveles geográficos y SCIAN.
-- Los 10 indicadores permanecen como columnas para lectura eficiente en DuckDB columnar.
{{
    config(
        post_hook=[
            "CREATE INDEX IF NOT EXISTS idx_{{ this.name }}_nivel_geo ON {{ this }} (nivel_geografico)",
            "CREATE INDEX IF NOT EXISTS idx_{{ this.name }}_nivel_scian ON {{ this }} (nivel_scian)",
            "CREATE INDEX IF NOT EXISTS idx_{{ this.name }}_sector ON {{ this }} (codigo_sector)",
            "CREATE INDEX IF NOT EXISTS idx_{{ this.name }}_cve_ent ON {{ this }} (cve_ent)",
        ]
    )
}}

select *
from {{ ref("int_ce__datos") }}

### Ejercicio 6.4: Tabla de hechos de presupuesto mensual (`fct_presupuesto_mensual.sql`)

Transforma las 12 columnas de meses en filas usando `UNPIVOT` de DuckDB. Cada fila es una combinación de año/entidad/fondo/mes.

El `CASE` convierte el nombre del mes a número para ordenamiento cronológico.

> **Documentación**: [UNPIVOT en DuckDB](https://duckdb.org/docs/sql/statements/unpivot)


In [ ]:
%%writefile mi_proyecto_dbt/models/mart/fct_presupuesto_mensual.sql
-- Presupuesto federal despivoteado por mes.
-- Grano: año × ramo × anexo × fondo × entidad × mes.
with
    fuente as (select * from {{ ref("int_dof__presupuesto") }}),

    despivoteado as (
        select *
        from
            fuente unpivot (
                monto for mes_nombre in (
                    enero,
                    febrero,
                    marzo,
                    abril,
                    mayo,
                    junio,
                    julio,
                    agosto,
                    septiembre,
                    octubre,
                    noviembre,
                    diciembre
                )
            )
    )

select
    d.anio,
    d.cve_ent,
    e.nombre_entidad,
    d.ramo,
    d.anexo,
    d.fondo,
    d.mes_nombre,
    -- Número de mes para ordenar cronológicamente
    case
        d.mes_nombre
        when 'enero'
        then 1
        when 'febrero'
        then 2
        when 'marzo'
        then 3
        when 'abril'
        then 4
        when 'mayo'
        then 5
        when 'junio'
        then 6
        when 'julio'
        then 7
        when 'agosto'
        then 8
        when 'septiembre'
        then 9
        when 'octubre'
        then 10
        when 'noviembre'
        then 11
        when 'diciembre'
        then 12
    end as mes_numero,
    d.monto
from despivoteado as d
left join {{ ref("dim_entidades") }} as e on d.cve_ent = e.cve_ent
order by d.anio, d.cve_ent, d.ramo, mes_numero

### Ejercicio 6.5: Tests de marts (`_mart__models.yml`)

Tests más rigurosos para la capa final:
- `relationships`: verifica integridad referencial entre tablas
- `accepted_values`: valida dominios de columnas
- `severity: warn`: permite que el test falle sin detener el pipeline

> **Documentación**: [Test `relationships`](https://docs.getdbt.com/reference/resource-properties/data-tests#relationships) · [Severidad de tests](https://docs.getdbt.com/reference/resource-configs/severity)


In [ ]:
%%writefile mi_proyecto_dbt/models/mart/_mart__models.yml
version: 2

models:
  # -- Dimensiones --
  - name: dim_entidades
    description: "Dimensión de 5 entidades federativas del tutorial"
    columns:
      - name: cve_ent
        description: "Clave de entidad (2 caracteres)"
        tests:
          - unique
          - not_null
      - name: nombre_entidad
        description: "Nombre oficial de la entidad"
        tests:
          - not_null

  - name: dim_actividades_economicas
    description: "Jerarquía SCIAN con navegación padre-hijo para drill-down"
    columns:
      - name: codigo
        description: "Código SCIAN de la actividad"
        tests:
          - unique
          - not_null
      - name: descripcion
        description: "Descripción de la actividad económica"
        tests:
          - not_null
      - name: nivel
        description: "Nivel en la jerarquía: sector, subsector, rama"
        tests:
          - not_null
          - accepted_values:
              arguments:
                values: ["sector", "subsector", "rama"]
      - name: codigo_padre
        description: "Código del nivel superior (NULL para sectores)"
      - name: codigo_sector
        description: "Código del sector (maneja compuesto: 31-33)"
        tests:
          - not_null

  - name: dim_ce_indicadores
    description: "Diccionario de indicadores del Censo Económico con bandera de indicador clave"
    columns:
      - name: nombre_columna
        description: "Nombre del indicador (minúsculas)"
        tests:
          - unique
          - not_null

  # -- Hechos --
  - name: fct_censo_economico
    description: >
      Censo Económico pivotado con todos los niveles geográficos y SCIAN.
      Los 10 indicadores permanecen como columnas.
      Grano: anio × nivel_geografico × cve_ent × cve_mun × codigo_actividad.
    columns:
      - name: censo_economico_sk
        description: "Clave sustituta sobre el grano completo"
        tests:
          - unique
          - not_null
      - name: anio
        tests:
          - not_null
      - name: nivel_geografico
        description: "Nivel geográfico: nacional, entidad o municipio"
        tests:
          - not_null
          - accepted_values:
              arguments:
                values: ["nacional", "entidad", "municipio"]
      - name: nivel_scian
        description: "Nivel SCIAN: sector, subsector, rama"
        tests:
          - not_null
          - accepted_values:
              arguments:
                values: ["sector", "subsector", "rama"]
      - name: codigo_actividad
        tests:
          - not_null

  - name: fct_presupuesto_mensual
    description: "Presupuesto federal despivoteado por mes. Grano: año × ramo × anexo × fondo × entidad × mes"
    columns:
      - name: mes_nombre
        description: "Nombre del mes en español (enero-diciembre)"
        tests:
          - not_null
          - accepted_values:
              arguments:
                values: ['enero', 'febrero', 'marzo', 'abril', 'mayo', 'junio',
                         'julio', 'agosto', 'septiembre', 'octubre', 'noviembre', 'diciembre']
      - name: mes_numero
        description: "Número del mes (1-12) para ordenamiento cronológico"
        tests:
          - not_null
      - name: cve_ent
        description: "Clave de entidad"
        tests:
          - not_null
          - relationships:
              arguments:
                to: ref('dim_entidades')
                field: cve_ent
      - name: nombre_entidad
        description: "Nombre de la entidad federativa"
        tests:
          - not_null
      - name: anio
        tests:
          - not_null
      - name: monto
        description: "Monto asignado para el mes"
        tests:
          - not_null:
              config:
                severity: warn

### Ejecutar un build completo (sin seed ni dim_ce_indicadores todavía)

Ejecutamos todo excepto `dim_ce_indicadores` que depende del seed que crearemos en la siguiente sección.

In [ ]:
!cd mi_proyecto_dbt && uv run dbt build --exclude dim_ce_indicadores

---
## Sección 7: Macros, Seeds y Jinja avanzado

### Macros
Las macros en dbt son funciones Jinja reutilizables. Permiten encapsular lógica SQL repetitiva.

### Seeds
Los seeds son archivos CSV que dbt carga como tablas. Son útiles para:
- Datos de referencia pequeños
- Mapeos manuales
- Configuraciones que cambian raramente

> **Documentación**: [Macros](https://docs.getdbt.com/docs/build/jinja-macros#macros) · [Seeds](https://docs.getdbt.com/docs/build/seeds) · [Comando `dbt seed`](https://docs.getdbt.com/reference/commands/seed)


### Ejercicio 7.1: Macro `normalizar_texto`

Refactorizamos la cadena de `regexp_replace` del staging DOF a una macro reutilizable. Así cualquier modelo puede normalizar texto con `{{ normalizar_texto('nombre_columna') }}`.

> **Documentación**: [Escribir macros propias](https://docs.getdbt.com/docs/build/jinja-macros#macros)


In [ ]:
%%writefile mi_proyecto_dbt/macros/normalizar_texto.sql
-- Macro para normalizar texto: minúsculas, sin acentos, sin asteriscos,
-- espacios normalizados, ñ → ni.
-- Uso: {{ normalizar_texto('nombre_columna') }}
{% macro normalizar_texto(columna) %}
regexp_replace(
    regexp_replace(
        regexp_replace(
            regexp_replace(
                regexp_replace(
                    regexp_replace(
                        regexp_replace(
                            regexp_replace(
                                lower(regexp_replace(trim({{ columna }}), '\s+', ' ', 'g')),
                                '^\*+\s*', ''
                            ),
                            'á', 'a'
                        ),
                        'é', 'e'
                    ),
                    'í', 'i'
                ),
                'ó', 'o'
            ),
            'ú', 'u'
        ),
        'ü', 'u'
    ),
    'ñ', 'ni'
)
{% endmacro %}

Ahora reescribimos `stg_dof__presupuesto.sql` para usar la macro en vez del bloque Jinja inline:

In [ ]:
%%writefile mi_proyecto_dbt/models/staging/stg_dof__presupuesto.sql
-- Staging del presupuesto DOF: normaliza nombres de entidad a cve_ent
-- mediante la macro normalizar_texto y un mapeo inline de 5 estados.
-- Las filas que no mapean (junk: totales, consolidados) quedan con cve_ent NULL.
with
    fuente as (select * from {{ source("tutorial", "dof_presupuesto") }}),

    -- Mapeo de nombres normalizados de entidad → cve_ent
    mapeo_entidades(nombre_normalizado, cve_ent) as (
        values
            ('ciudad de mexico', '09'),
            ('guanajuato', '11'),
            ('jalisco', '14'),
            ('nuevo leon', '19'),
            ('puebla', '21')
    ),

    con_normalizacion as (
        select *, {{ normalizar_texto('entidad') }} as entidad_normalizada from fuente
    ),

    mapeado as (
        select
            f.id as id_fuente,
            f.anio,
            f.ramo,
            f.anexo,
            trim(f.fondo) as fondo,
            trim(f.entidad) as entidad_original,
            f.entidad_normalizada,
            m.cve_ent,
            f.es_total,
            f.es_consolidado,
            f.anual,
            f.enero,
            f.febrero,
            f.marzo,
            f.abril,
            f.mayo,
            f.junio,
            f.julio,
            f.agosto,
            f.septiembre,
            f.octubre,
            f.noviembre,
            f.diciembre
        from con_normalizacion as f
        left join mapeo_entidades as m on f.entidad_normalizada = m.nombre_normalizado
    )

select *
from mapeado

### Ejercicio 7.2: Seed de indicadores clave (`indicadores_clave.csv`)

Este CSV pequeño marca cuáles de los 10 indicadores del CE son "clave" para el análisis. dbt lo cargará como una tabla con `dbt seed`.

> **Documentación**: [Seeds](https://docs.getdbt.com/docs/build/seeds) · [Configuración de seeds](https://docs.getdbt.com/reference/seed-configs)


In [ ]:
%%writefile mi_proyecto_dbt/seeds/indicadores_clave.csv
nombre_columna,es_clave
ue,true
h001a,true
a111a,true
j000a,true
k000a,true

In [ ]:
# Cargar el seed como tabla
!cd mi_proyecto_dbt && uv run dbt seed

### Ejercicio 7.3: Dimensión de indicadores (`dim_ce_indicadores.sql`)

Este modelo combina el diccionario de datos (source) con el seed de indicadores clave:
- Filtra filas de metadatos que no son indicadores
- Toma la descripción del año más reciente por indicador
- Extrae la descripción corta y la unidad de medida del texto entre paréntesis
- Marca indicadores clave según el seed

In [ ]:
%%writefile mi_proyecto_dbt/models/mart/dim_ce_indicadores.sql
-- Diccionario de indicadores del Censo Económico.
-- Usa la descripción del año más reciente y marca indicadores clave via seed.
with
    diccionario as (
        select
            lower(nombre_columna) as nombre_columna,
            descripcion,
            tipo_dato,
            anio
        from {{ source("tutorial", "ce_diccionarios_datos") }}
        -- Filtrar filas de metadatos (no son indicadores)
        where nombre_columna not in ('ENTIDAD', 'MUNICIPIO', 'CODIGO', 'ID_ESTRATO', 'E000')
    ),

    -- Tomar la descripción del año más reciente por indicador
    con_rango as (
        select
            *, row_number() over (partition by nombre_columna order by anio desc) as rn
        from diccionario
    ),

    mas_reciente as (select * from con_rango where rn = 1),

    -- Indicadores clave desde el seed
    clave as (
        select nombre_columna
        from {{ ref("indicadores_clave") }}
        where es_clave = 'true'
    )

select
    mr.nombre_columna,
    mr.descripcion,
    -- Descripción corta: todo antes del primer paréntesis
    trim(regexp_extract(mr.descripcion, '^([^(]+)', 1)) as descripcion_corta,
    -- Unidad de medida extraída del paréntesis
    nullif(trim(regexp_extract(mr.descripcion, '\(([^)]+)\)', 1)), '') as unidad,
    mr.tipo_dato,
    mr.anio as anio_descripcion,
    case
        when ic.nombre_columna is not null then true else false
    end as es_indicador_clave
from mas_reciente as mr
left join clave as ic on mr.nombre_columna = ic.nombre_columna
order by mr.nombre_columna

### Build completo con la macro refactorizada y el nuevo modelo

In [ ]:
!cd mi_proyecto_dbt && uv run dbt build

---
## Sección 8: Documentación y cierre

dbt genera documentación automática a partir de:
- Las descripciones en los archivos YAML
- El linaje entre modelos (DAG)
- Las columnas y tests definidos

### Ejercicio 8.1: Completar las descripciones

En un proyecto real, este es el momento para revisar que todas las columnas de los archivos `_*__models.yml` tengan descripciones completas. Los archivos que creamos ya incluyen descripciones, pero en la práctica iterarías sobre ellos.

> **Documentación**: [Documentación en dbt](https://docs.getdbt.com/docs/collaborate/documentation) · [Comando `dbt docs generate`](https://docs.getdbt.com/reference/commands/cmd-docs) · [Explorar el lineage graph](https://docs.getdbt.com/docs/collaborate/explore-projects)


### Generar la documentación

In [ ]:
!cd mi_proyecto_dbt && uv run dbt docs generate

In [ ]:
# Servir la documentación en un puerto local (se abre en el navegador)
# Ejecutar en una terminal separada o usar el botón de parar del notebook para detenerlo
# !cd mi_proyecto_dbt && uv run dbt docs serve --port 8082

### Build final completo

> **Documentación**: [Comando `dbt build`](https://docs.getdbt.com/reference/commands/build)


In [ ]:
# Build completo: seeds + modelos + tests
!cd mi_proyecto_dbt && uv run dbt build

### Verificación final: consultar las tablas resultantes

In [ ]:
import duckdb

con = duckdb.connect("datos_tutorial.duckdb", read_only=True)

# Esquemas creados por dbt
print("=== Esquemas ===")
con.sql("SELECT schema_name FROM information_schema.schemata ORDER BY schema_name").show()

# Tablas en el esquema mart
print("\n=== Tablas en mart ===")
con.sql("""
    SELECT table_name
    FROM information_schema.tables
    WHERE table_schema = 'mart'
    ORDER BY table_name
""").show()

In [ ]:
# Consulta de ejemplo: Top 5 actividades por valor agregado censal bruto (a111a)
con.sql("""
    SELECT
        f.codigo_actividad,
        d.descripcion,
        f.nivel_scian,
        SUM(f.a111a) AS valor_agregado_total
    FROM mart.fct_censo_economico AS f
    JOIN mart.dim_actividades_economicas AS d ON f.codigo_actividad = d.codigo
    WHERE f.nivel_geografico = 'nacional'
      AND f.nivel_scian = 'sector'
      AND f.anio = 2024
    GROUP BY f.codigo_actividad, d.descripcion, f.nivel_scian
    ORDER BY valor_agregado_total DESC
""").show()

In [ ]:
# Consulta de ejemplo: Presupuesto mensual de Jalisco por fondo
con.sql("""
    SELECT
        fondo,
        mes_nombre,
        mes_numero,
        monto
    FROM mart.fct_presupuesto_mensual
    WHERE nombre_entidad = 'Jalisco'
      AND anio = 2024
    ORDER BY fondo, mes_numero
    LIMIT 15
""").show()

In [ ]:
# Consulta de ejemplo: Indicadores clave del diccionario
con.sql("""
    SELECT nombre_columna, descripcion_corta, unidad, es_indicador_clave
    FROM mart.dim_ce_indicadores
    ORDER BY es_indicador_clave DESC, nombre_columna
""").show()

In [ ]:
con.close()

---
## Resumen

En este tutorial construimos un pipeline dbt completo con arquitectura medallón:

| Capa | Modelos | Qué hace |
|------|---------|----------|
| **Sources** | `sources.yml` | Declara 5 tablas crudas del esquema `main` |
| **Staging** | 4 modelos `stg_*` | Limpieza: `TRIM`, `NULLIF`, normalización de texto |
| **Intermediate** | 2 modelos `int_*` | Joins, claves sustitutas, filtrado de basura |
| **Mart** | 3 dimensiones + 2 hechos | Tablas listas para BI y análisis |

### Conceptos cubiertos

- **Configuración**: `dbt_project.yml`, `profiles.yml`, `packages.yml`
- **Sources**: declaración de tablas crudas con `{{ source() }}`
- **Modelos**: SQL con Jinja, CTEs, `{{ ref() }}`
- **Materialización**: views vs tables, esquemas personalizados
- **Tests**: `unique`, `not_null`, `accepted_values`, `relationships`
- **Macros**: `normalizar_texto`, `generate_schema_name`
- **Seeds**: datos de referencia desde CSV
- **dbt_utils**: `generate_surrogate_key`
- **DuckDB**: `UNPIVOT`, `regexp_replace`, `regexp_extract`
- **Documentación**: `dbt docs generate`

### Próximos pasos

- Agregar modelos incrementales para tablas que crecen
- Implementar snapshots para capturar cambios en dimensiones
- Añadir data freshness checks a las sources
- Integrar con un orquestador (Airflow, GitHub Actions)
- Desplegar con CI/CD